In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

# Sklearn models
> Use any scikit-learn model for forecasting

statsforecast supports providing scikit-learn models through the `statsforecast.models.SklearnModel` wrapper. This can help you leverage feature engineering and train one model per serie, which can sometimes be better than training a single global model (as in mlforecast).

## Data setup

In [ ]:
import os
from functools import partial

from datasetsforecast.m4 import M4, M4Info
from sklearn.linear_model import Lasso, Ridge
from utilsforecast.feature_engineering import pipeline, trend, fourier
from utilsforecast.plotting import plot_series

from statsforecast import StatsForecast
from statsforecast.models import SklearnModel
from statsforecast.utils import ConformalIntervals

In [ ]:
# this makes it so that the outputs of the predict methods have the id as a column 
# instead of as the index
os.environ['NIXTLA_ID_AS_COL'] = '1'

In [ ]:
group = 'Hourly'
season_length = M4Info[group].seasonality
horizon = M4Info[group].horizon
data, *_ = M4.load('data', group)
data['ds'] = data['ds'].astype('int64')
valid = data.groupby('unique_id').tail(horizon).copy()
train = data.drop(valid.index)
train.head()

## Generating features

The utilsforecast library [provides some utilies for feature engineering](https://nixtlaverse.nixtla.io/utilsforecast/feature_engineering.html).

In [ ]:
train_features, valid_features = pipeline(
    train,
    features=[
        trend,        
        partial(fourier, season_length=season_length, k=10),  # 10 fourier terms
    ],
    freq=1,
    h=horizon,
)
train_features.head()

## Forecasting

In [ ]:
sf = StatsForecast(
    models=[
        SklearnModel(Lasso()),
        SklearnModel(Ridge()),
    ],
    freq=1,
)
preds = sf.forecast(
    df=train_features,
    h=horizon,
    X_df=valid_features,
    prediction_intervals=ConformalIntervals(n_windows=4, h=horizon),
    level=[95],
)
plot_series(train, preds, level=[95], palette='tab20b', max_ids=4)